# 🎯 Object Detection: YOLO, RT-DETR & YOLO-World

В этом воркшопе разберём несколько современных архитектур детекции из экосистемы **Ultralytics**:

| Архитектура | Тип | Ключевая идея |
|:-----------:|:---:|:-------------|
| **YOLO12** | One-stage CNN | Attention-based, быстрый и точный |
| **RT-DETR** | Transformer | Detection Transformer без NMS, real-time |
| **YOLO-World** | Open-vocabulary | Zero-shot детекция по текстовому запросу |

### Задачи YOLO-моделей:
- `detect` — детекция объектов (bounding boxes)
- `segment` — инстанс-сегментация
- `pose` — определение ключевых точек
- `obb` — oriented bounding boxes
- `classify` — классификация изображений

<img width="900" src="https://yolov8.org/wp-content/uploads/2024/01/How-Does-YOLOv8-Work-2-1024x536.webp">


---
## 📦 Установка и настройка

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import ssl
ssl._create_default_https_context = ssl._create_stdlib_context


In [ ]:
try:
    import ultralytics
except ImportError:
    %pip install ultralytics -q
    import ultralytics

ultralytics.checks()
print(f"Ultralytics version: {ultralytics.__version__}")


In [ ]:
import os, cv2, glob, yaml, json
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from shutil import copyfile
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
from torchvision.datasets.utils import download_and_extract_archive

from ultralytics import YOLO, RTDETR
from ultralytics import settings

# Отключаем внешнее логирование
settings.update({
    "wandb": False, "tensorboard": False, "clearml": False,
    "mlflow": False, "neptune": False, "comet": False,
    "dvc": False, "hub": False, "raytune": False
})

print("✅ Все импорты выполнены")


---
## 🔍 Часть 1. Задачи YOLO (YOLOv8n — обзор)

Запустим **YOLOv8 nano** на одном изображении для всех типов задач.
Nano — самая маленькая и быстрая версия, удобна для демонстрации.


In [ ]:
TEST_URL = "https://ultralytics.com/images/zidane.jpg"

YOLO_TASKS = {
    "detect":   "yolov8n.pt",
    "segment":  "yolov8n-seg.pt",
    "pose":     "yolov8n-pose.pt",
    "classify": "yolov8n-cls.pt",
}

for task, weights in YOLO_TASKS.items():
    model = YOLO(weights)
    res   = model.predict(TEST_URL, imgsz=640, verbose=False)
    res[0].save(filename=f"result_{task}.jpg")
    print(f"✅ {task:10s} → {weights}")


In [ ]:
fig, axes = plt.subplots(1, len(YOLO_TASKS), figsize=(20, 5))
for ax, (task, _) in zip(axes, YOLO_TASKS.items()):
    img = cv2.imread(f"result_{task}.jpg")[:, :, ::-1]
    ax.imshow(img); ax.set_title(task, fontsize=13); ax.axis('off')
plt.suptitle("YOLOv8n — все задачи", fontsize=15)
plt.tight_layout(); plt.show()


---
## 🪖 Часть 2. Датасет — Helmet Detection

Датасет для детекции касок на промышленных объектах.  
**2 класса:** `helmet` (0) и `head` (1, без каски).

### Формат разметки YOLO:
```
{class_id}  {x_center}  {y_center}  {width}  {height}
```
Все значения — нормализованные (0..1 относительно размера изображения).

| Пример | Что означает |
|:------:|:--------:|
| `0 0.51 0.33 0.12 0.18` | Каска, центр (51%, 33%), размер 12%×18% |
| `1 0.72 0.45 0.10 0.15` | Голова без каски |


In [ ]:
url = "https://github.com/MVRonkin/Deep-Learning-Foundation-Course/raw/refs/heads/main/DLUtilits/Helmet.zip"

archive_name = url.split('/')[-1]
root_directory = os.path.join(os.getcwd(), 'datasets')
dataset_directory = os.path.splitext(os.path.join(root_directory, archive_name))[0]

if not os.path.exists(dataset_directory):
    download_and_extract_archive(url, root_directory)
    print("✅ Датасет скачан")
else:
    print("✅ Датасет уже есть")

print("\nСодержимое датасета:", os.listdir(dataset_directory))

# Считаем изображения
for split in ['train', 'val', 'test']:
    img_dir = Path(dataset_directory) / 'images' / split
    if img_dir.exists():
        n = len(list(img_dir.iterdir()))
        print(f"  {split:5s}: {n} изображений")


In [ ]:
# Создаём/обновляем data.yaml
data_yaml = Path(dataset_directory) / 'data.yaml'
data_cfg = {
    'train': str(Path(dataset_directory) / 'images' / 'train'),
    'val':   str(Path(dataset_directory) / 'images' / 'val'),
    'test':  str(Path(dataset_directory) / 'images' / 'test'),
    'nc': 2,
    'names': ['helmet', 'head']
}
with open(data_yaml, 'w') as f:
    yaml.safe_dump(data_cfg, f)

print("data.yaml:")
print(yaml.dump(data_cfg))


In [ ]:
def plot_yolo_annotations(img_folder, label_folder, img_ids=None,
                          class_names=None, n_cols=4):
    """Визуализация YOLO-разметки с bounding boxes."""
    all_imgs = sorted(Path(img_folder).glob("*.jpg")) +                sorted(Path(img_folder).glob("*.png"))
    if img_ids is None:
        img_ids = list(range(min(n_cols, len(all_imgs))))

    COLORS = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']

    fig, axes = plt.subplots(1, len(img_ids), figsize=(5*len(img_ids), 5))
    if len(img_ids) == 1:
        axes = [axes]

    for ax, idx in zip(axes, img_ids):
        img_path   = all_imgs[idx]
        label_path = Path(label_folder) / (img_path.stem + ".txt")

        img = Image.open(img_path)
        W, H = img.size
        ax.imshow(img)

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if len(parts) < 5:
                        continue
                    cls_id, xc, yc, w, h = parts[:5]
                    cls_id = int(cls_id)
                    color  = COLORS[cls_id % len(COLORS)]
                    name   = class_names[cls_id] if class_names else f"cls{cls_id}"

                    x1 = (xc - w/2) * W
                    y1 = (yc - h/2) * H
                    rect = Rectangle((x1, y1), w*W, h*H,
                                     linewidth=2, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)
                    ax.text(x1, y1 - 6, name, color=color, fontsize=9,
                            bbox=dict(facecolor='white', alpha=0.7, pad=1))
        ax.set_title(img_path.name[:20], fontsize=9)
        ax.axis('off')

    plt.suptitle("Примеры датасета с разметкой", fontsize=13)
    plt.tight_layout(); plt.show()


plot_yolo_annotations(
    Path(dataset_directory) / 'images' / 'train',
    Path(dataset_directory) / 'labels' / 'train',
    img_ids=[0, 1, 2, 3],
    class_names=['helmet', 'head']
)


In [ ]:
# ── Статистика датасета ────────────────────────────────────────────────────────
def dataset_stats(dataset_dir, class_names):
    """Считает количество объектов каждого класса в каждом split."""
    stats = {}
    for split in ['train', 'val', 'test']:
        label_dir = Path(dataset_dir) / 'labels' / split
        if not label_dir.exists():
            continue
        counts = {c: 0 for c in class_names}
        n_imgs = 0
        for lf in label_dir.glob("*.txt"):
            n_imgs += 1
            for line in lf.read_text().strip().split('\n'):
                if line:
                    cls_id = int(line.split()[0])
                    if cls_id < len(class_names):
                        counts[class_names[cls_id]] += 1
        stats[split] = {'images': n_imgs, **counts}

    df = pd.DataFrame(stats).T
    print("Статистика датасета:")
    print(df.to_string())

    # Визуализация
    splits = list(stats.keys())
    x = np.arange(len(splits))
    w = 0.35
    fig, ax = plt.subplots(figsize=(7, 4))
    for i, cls in enumerate(class_names):
        vals = [stats[s].get(cls, 0) for s in splits]
        ax.bar(x + i*w - w/2, vals, w, label=cls,
               color=['#2ecc71', '#e74c3c'][i], alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(splits)
    ax.set_ylabel("Количество объектов")
    ax.set_title("Распределение классов по split-ам")
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

dataset_stats(dataset_directory, ['helmet', 'head'])


---
## 🚀 Часть 3. Обучение YOLO12

**YOLO12** — одна из последних версий YOLO от Ultralytics.  
Использует **attention-механизмы** вместо классических свёрток, что даёт лучшее качество при сопоставимой скорости.

### Размеры моделей (nano → xlarge):
| Модель | Параметры | mAP50-95 (COCO) | Скорость |
|:------:|:---------:|:---------------:|:--------:|
| YOLO12n | ~2.6M | 40.6 | ⚡⚡⚡⚡ |
| YOLO12s | ~9.3M | 48.0 | ⚡⚡⚡ |
| YOLO12m | ~20M  | 52.5 | ⚡⚡ |
| YOLO12l | ~27M  | 53.7 | ⚡ |


In [ ]:
def save_best_weights(results, fname='best.pt'):
    """Копирует best.pt из папки runs в текущую директорию."""
    src = Path(str(results.save_dir)) / 'weights' / 'best.pt'
    dst = Path(os.getcwd()) / fname
    copyfile(src, dst)
    print(f"✅ Веса сохранены: {dst}")
    return str(dst)


In [ ]:
# ── Обучение YOLO12n ─────────────────────────────────────────────────────────
EPOCHS   = 10
IMG_SIZE = 416
BATCH    = 8

model_yolo12 = YOLO("yolo12n.pt")

results_yolo12 = model_yolo12.train(
    data    = str(data_yaml),
    imgsz   = IMG_SIZE,
    epochs  = EPOCHS,
    batch   = BATCH,
    lr0     = 0.001,
    amp     = False,
    workers = 0,
    project = "runs/helmet",
    name    = "yolo12n",
    verbose = False,
)

yolo12_weights = save_best_weights(results_yolo12, 'yolo12n_best.pt')
print(f"\nmAP@0.5:     {results_yolo12.box.map50*100:.2f}%")
print(f"mAP@0.5:0.95: {results_yolo12.box.map*100:.2f}%")


In [ ]:
# ── Просмотр тренировочных батчей ─────────────────────────────────────────────
save_dir = Path(str(results_yolo12.save_dir))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, fname in zip(axes, ['train_batch0.jpg', 'val_batch0_pred.jpg']):
    fpath = save_dir / fname
    if fpath.exists():
        ax.imshow(cv2.imread(str(fpath))[:, :, ::-1])
    ax.set_title(fname); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ── Кривые обучения из results.csv ───────────────────────────────────────────
results_csv = save_dir / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Loss
    for col in ['train/box_loss', 'val/box_loss']:
        if col in df.columns:
            axes[0].plot(df[col], label=col.split('/')[0])
    axes[0].set_title('Box Loss'); axes[0].legend(); axes[0].grid(True)

    # mAP
    for col in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)']:
        if col in df.columns:
            axes[1].plot(df[col]*100, label=col)
    axes[1].set_title('mAP (%)'); axes[1].legend(); axes[1].grid(True)

    # Precision / Recall
    for col in ['metrics/precision(B)', 'metrics/recall(B)']:
        if col in df.columns:
            axes[2].plot(df[col]*100, label=col.split('/')[1].split('(')[0])
    axes[2].set_title('Precision & Recall (%)'); axes[2].legend(); axes[2].grid(True)

    plt.suptitle('YOLO12n — кривые обучения', fontsize=13)
    plt.tight_layout(); plt.show()


---
## ⚡ Часть 4. RT-DETR — Real-Time Detection Transformer

**RT-DETR** (Real-Time Detection Transformer) — принципиально другая архитектура:

| Свойство | YOLO | RT-DETR |
|:--------:|:----:|:-------:|
| Тип | CNN + Head | Vision Transformer |
| NMS | Требуется | **Не нужен** (end-to-end) |
| Скорость | Очень высокая | Высокая |
| Точность | Высокая | **Выше при схожей скорости** |
| Backbone | CSPDarkNet | ResNet / HGNetv2 |

### Почему без NMS?
RT-DETR использует **механизм cross-attention** между query-векторами и признаками изображения.  
Каждый query «отвечает» за один объект → нет дублирующихся боксов → NMS не нужен.

```
Image → Backbone (ResNet) → Encoder (Transformer) 
      → Decoder (Cross-Attention queries) → Boxes + Classes
```


In [ ]:
# ── Обучение RT-DETR-l ──────────────────────────────────────────────────────
# RT-DETR доступен в двух размерах: rtdetr-l (large) и rtdetr-x (extra large)

model_rtdetr = RTDETR("rtdetr-l.pt")

print("RT-DETR архитектура (краткая):")
print(f"  Параметров: {sum(p.numel() for p in model_rtdetr.model.parameters()):,}")


In [ ]:
results_rtdetr = model_rtdetr.train(
    data    = str(data_yaml),
    imgsz   = IMG_SIZE,
    epochs  = EPOCHS,
    batch   = BATCH,
    lr0     = 0.0001,       # RT-DETR требует меньший LR
    amp     = False,
    workers = 0,
    project = "runs/helmet",
    name    = "rtdetr_l",
    verbose = False,
)

rtdetr_weights = save_best_weights(results_rtdetr, 'rtdetr_best.pt')
print(f"\nmAP@0.5:     {results_rtdetr.box.map50*100:.2f}%")
print(f"mAP@0.5:0.95: {results_rtdetr.box.map*100:.2f}%")


In [ ]:
# ── Сравнение кривых обучения YOLO12 vs RT-DETR ──────────────────────────────
def load_history(save_dir):
    csv_path = Path(str(save_dir)) / 'results.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        return df
    return None

df_yolo  = load_history(results_yolo12.save_dir)
df_rtdet = load_history(results_rtdetr.save_dir)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for df, label, color in [(df_yolo, 'YOLO12n', 'steelblue'),
                          (df_rtdet, 'RT-DETR-l', 'coral')]:
    if df is None:
        continue
    col_map50 = 'metrics/mAP50(B)'
    col_loss  = 'val/box_loss'
    if col_map50 in df.columns:
        axes[0].plot(df[col_map50]*100, label=label, color=color)
    if col_loss in df.columns:
        axes[1].plot(df[col_loss], label=label, color=color)

axes[0].set_title('mAP@0.5 (%)')
axes[1].set_title('Val Box Loss')
for ax in axes:
    ax.legend(); ax.grid(True)

plt.suptitle('YOLO12n vs RT-DETR-l', fontsize=13)
plt.tight_layout(); plt.show()


---
## 🌍 Часть 5. YOLO-World — Open-Vocabulary Detection

**YOLO-World** — zero-shot детектор: обнаруживает **любые объекты по текстовому описанию**, без дообучения.

### Как это работает?
1. Изображение кодируется CNN-энкодером
2. Текстовые запросы кодируются CLIP-подобным текстовым энкодером
3. Визуальные и текстовые эмбеддинги сравниваются → детекция

```
Изображение → Image Encoder ──┐
                               ├─ Matching → Boxes
Текст ("helmet", "car") → Text Encoder ──┘
```

### Применения:
- Быстрый прототип без сбора датасета
- Детекция редких классов
- Интерактивный поиск объектов


In [ ]:
from ultralytics import YOLOWorld

# Загружаем pretrained YOLO-World (small)
model_world = YOLOWorld("yolov8s-world.pt")
print("✅ YOLO-World загружен")


In [ ]:
# ── Zero-shot детекция на нашем датасете ─────────────────────────────────────
# Указываем классы ТЕКСТОМ — никакого обучения не нужно!
model_world.set_classes(["safety helmet", "human head without helmet"])

test_imgs = sorted((Path(dataset_directory) / 'images' / 'test').glob("*.jpg"))[:3]

fig, axes = plt.subplots(1, len(test_imgs), figsize=(6*len(test_imgs), 5))
for ax, img_path in zip(axes, test_imgs):
    results = model_world.predict(str(img_path), conf=0.2, verbose=False)
    img = cv2.imread(str(img_path))[:, :, ::-1].copy()

    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls  = int(box.cls)
        conf = float(box.conf)
        color = (46, 204, 113) if cls == 0 else (231, 76, 60)  # green / red
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = f"{results[0].names[cls][:12]} {conf:.2f}"
        cv2.putText(img, label, (x1, max(y1-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

    ax.imshow(img); ax.set_title(img_path.name[:20]); ax.axis('off')

plt.suptitle("YOLO-World Zero-Shot (без дообучения!)", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── YOLO-World с разными текстовыми запросами ────────────────────────────────
# Попробуем разные формулировки на одном изображении
QUERIES = [
    ["helmet", "head"],
    ["hard hat", "person without hat"],
    ["protective equipment", "unprotected worker"],
]

sample_img = str(test_imgs[0])
fig, axes = plt.subplots(1, len(QUERIES), figsize=(6*len(QUERIES), 5))

for ax, classes in zip(axes, QUERIES):
    model_world.set_classes(classes)
    res = model_world.predict(sample_img, conf=0.15, verbose=False)

    img = cv2.imread(sample_img)[:, :, ::-1].copy()
    for box in res[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls  = int(box.cls)
        color = (46, 204, 113) if cls == 0 else (231, 76, 60)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        ax.text(x1, y1-4, f"{classes[cls][:10]} {float(box.conf):.2f}",
                color='white', fontsize=7,
                bbox=dict(facecolor=('#2ecc71' if cls==0 else '#e74c3c'), alpha=0.8, pad=1))
    ax.imshow(img)
    ax.set_title(f'"{classes[0]}" vs "{classes[1]}"', fontsize=8)
    ax.axis('off')

plt.suptitle("YOLO-World: влияние формулировки запроса", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ── YOLO-World: fine-tune на нашем датасете ──────────────────────────────────
# Дообучение значительно улучшает точность для конкретных классов

model_world_ft = YOLOWorld("yolov8s-world.pt")
model_world_ft.set_classes(["helmet", "head"])

results_world = model_world_ft.train(
    data    = str(data_yaml),
    imgsz   = IMG_SIZE,
    epochs  = EPOCHS,
    batch   = BATCH,
    lr0     = 0.0005,
    amp     = False,
    workers = 0,
    project = "runs/helmet",
    name    = "yolo_world_ft",
    verbose = False,
)

world_weights = save_best_weights(results_world, 'yolo_world_best.pt')
print(f"\nmAP@0.5:     {results_world.box.map50*100:.2f}%")
print(f"mAP@0.5:0.95: {results_world.box.map*100:.2f}%")


---
## 📊 Часть 6. Сравнение моделей

Сравниваем все три обученные модели по метрикам и скорости.


In [ ]:
import time

def benchmark_model(model, img_paths, n_warmup=3):
    """Измеряет скорость инференса (мс/изображение)."""
    # Прогрев
    for p in img_paths[:n_warmup]:
        model.predict(str(p), verbose=False)

    # Измерение
    t0 = time.perf_counter()
    for p in img_paths:
        model.predict(str(p), verbose=False)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    return elapsed_ms / len(img_paths)


def evaluate_model_map(weights_path, model_cls=YOLO):
    """Запускает val и возвращает метрики."""
    m = model_cls(weights_path)
    res = m.val(data=str(data_yaml), verbose=False)
    return {
        'mAP50':    res.box.map50 * 100,
        'mAP50-95': res.box.map   * 100,
    }


test_imgs_list = sorted((Path(dataset_directory) / 'images' / 'test').glob("*.jpg"))[:20]
print(f"Оцениваем на {len(test_imgs_list)} тестовых изображениях...")


In [ ]:
# ── Запуск сравнения ─────────────────────────────────────────────────────────
comparison = {}

for name, weights, cls_ in [
    ("YOLO12n",          'yolo12n_best.pt', YOLO),
    ("RT-DETR-l",        'rtdetr_best.pt',  RTDETR),
    ("YOLO-World (FT)",  'yolo_world_best.pt', YOLOWorld),
]:
    if not Path(weights).exists():
        print(f"⚠️  {name}: файл {weights} не найден, пропускаем")
        continue
    try:
        metrics = evaluate_model_map(weights, cls_)
        m = cls_(weights)
        speed  = benchmark_model(m, test_imgs_list)
        comparison[name] = {**metrics, 'Speed (ms/img)': speed}
        print(f"{'✅':2s} {name:22s}  mAP50={metrics['mAP50']:.1f}%  "
              f"mAP50-95={metrics['mAP50-95']:.1f}%  {speed:.1f} ms/img")
    except Exception as e:
        print(f"❌ {name}: {e}")


In [ ]:
# ── Таблица и графики ─────────────────────────────────────────────────────────
if comparison:
    df_cmp = pd.DataFrame(comparison).T
    print("\n=== Итоговая таблица ===")
    print(df_cmp.round(2).to_string())

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    names  = list(comparison.keys())
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6'][:len(names)]

    for ax, metric in zip(axes, ['mAP50', 'mAP50-95', 'Speed (ms/img)']):
        vals = [comparison[n].get(metric, 0) for n in names]
        bars = ax.bar(names, vals, color=colors, alpha=0.85, edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{v:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax.set_title(metric); ax.grid(axis='y', alpha=0.3)
        ax.set_xticklabels(names, rotation=10, ha='right')
        if 'ms' in metric:
            ax.set_ylabel("Время (мс) ↓ меньше = лучше")
        else:
            ax.set_ylabel("% ↑ больше = лучше")

    plt.suptitle("Сравнение моделей: Helmet Detection", fontsize=14)
    plt.tight_layout(); plt.show()


---
## 🖼️ Часть 7. Визуализация предсказаний

Смотрим на предсказания всех моделей для одних и тех же изображений.


In [ ]:
def predict_and_draw(model, img_path, conf=0.3, color_map=None):
    """Возвращает изображение с нарисованными боксами."""
    res = model.predict(str(img_path), conf=conf, verbose=False)[0]
    img = cv2.imread(str(img_path))[:, :, ::-1].copy()

    default_colors = [(46,204,113), (231,76,60), (52,152,219), (243,156,18)]

    for box in res.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls  = int(box.cls)
        conf_val = float(box.conf)
        col  = color_map[cls] if color_map else default_colors[cls % len(default_colors)]
        cv2.rectangle(img, (x1, y1), (x2, y2), col, 2)
        label = f"{res.names.get(cls, cls)} {conf_val:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(img, (x1, max(y1-th-4, 0)), (x1+tw+2, y1), col, -1)
        cv2.putText(img, label, (x1+1, max(y1-2, th+2)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
    return img


# Загружаем модели для визуализации
vis_models = {}
for name, wpath, cls_ in [
    ("YOLO12n",         'yolo12n_best.pt',    YOLO),
    ("RT-DETR-l",       'rtdetr_best.pt',     RTDETR),
    ("YOLO-World (FT)", 'yolo_world_best.pt', YOLOWorld),
]:
    if Path(wpath).exists():
        vis_models[name] = cls_(wpath)

N_VIS = 3
sample_paths = test_imgs_list[:N_VIS]
n_cols = 1 + len(vis_models)  # GT + каждая модель

fig, axes = plt.subplots(N_VIS, n_cols, figsize=(4*n_cols, 4*N_VIS))

for row, img_path in enumerate(sample_paths):
    # Ground Truth
    from PIL import Image as PILImage
    axes[row, 0].imshow(PILImage.open(img_path))
    axes[row, 0].set_title("Original" if row == 0 else "")
    axes[row, 0].axis('off')

    for col, (mname, mmodel) in enumerate(vis_models.items(), start=1):
        pred_img = predict_and_draw(mmodel, img_path)
        axes[row, col].imshow(pred_img)
        axes[row, col].set_title(mname if row == 0 else "")
        axes[row, col].axis('off')

plt.suptitle("Сравнение предсказаний (conf≥0.3)", fontsize=14)
plt.tight_layout(); plt.show()


---
## 💾 Часть 8. Export модели и Submission

### Форматы экспорта:

| Формат | Для чего | Команда |
|:------:|:--------:|:-------:|
| `onnx` | Кросс-платформенный инференс | `model.export(format='onnx')` |
| `torchscript` | PyTorch production | `model.export(format='torchscript')` |
| `tflite` | Android / Edge | `model.export(format='tflite')` |
| `engine` | NVIDIA TensorRT | `model.export(format='engine')` |


In [ ]:
# ── Экспорт лучшей модели в ONNX ─────────────────────────────────────────────
best_model_path = 'yolo12n_best.pt'  # или 'rtdetr_best.pt'

if Path(best_model_path).exists():
    model_export = YOLO(best_model_path)
    export_path  = model_export.export(format='onnx', imgsz=IMG_SIZE)
    print(f"✅ ONNX модель: {export_path}")
    print(f"   Размер файла: {Path(export_path).stat().st_size / 1024:.0f} KB")


In [ ]:
# ── Функции для Submission ────────────────────────────────────────────────────

def round2(x, digits=2):
    return round(float(x), digits)


def run_inference(model, img_folder, n_imgs=None):
    """Запускает инференс на папке с изображениями."""
    paths = sorted(Path(img_folder).glob("*.jpg")) +             sorted(Path(img_folder).glob("*.png"))
    if n_imgs:
        paths = paths[:n_imgs]
    return paths, model.predict([str(p) for p in paths], save=False, verbose=False)


def build_submission(img_paths, results, conf_threshold=0.3):
    """Формирует список предсказаний в формате для сдачи."""
    outputs = []
    for img_path, res in zip(img_paths, results):
        img = Image.open(img_path)
        W, H = img.size
        record = {
            'name':    img_path.stem,
            'size':    [W, H],
            'predict': []
        }
        for box in res.boxes:
            if float(box.conf) < conf_threshold:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            record['predict'].append([
                int(box.cls),
                round2(float(box.conf)),
                round2(x1), round2(y1),
                round2(x2), round2(y2),
            ])
        outputs.append(record)
    return outputs


In [ ]:
# ── Генерация submission ──────────────────────────────────────────────────────
submit_folder = Path(dataset_directory) / 'images' / 'test'

model_final = YOLO('yolo12n_best.pt')  # лучшая модель
img_paths, predictions = run_inference(model_final, submit_folder)
outputs = build_submission(img_paths, predictions, conf_threshold=0.3)

# Сохраняем
student_name = 'ФИО'
filename = f'submit_{student_name}.json'
with open(filename, 'w') as f:
    json.dump(outputs, f, indent=4, ensure_ascii=False)

print(f"✅ Submission сохранён: {filename}")
print(f"   Изображений: {len(outputs)}")
total_boxes = sum(len(o['predict']) for o in outputs)
print(f"   Всего предсказаний: {total_boxes}")


In [ ]:
# ── Быстрая проверка submission ───────────────────────────────────────────────
with open(filename) as f:
    check = json.load(f)

print(f"Проверка файла {filename}:")
print(f"  Изображений: {len(check)}")
print(f"\nПример первой записи:")
print(json.dumps(check[0], indent=2, ensure_ascii=False)[:400])


---
## 📝 Задания

### Задание 1 — Сравните размеры YOLO12

Обучите две версии YOLO12 и сравните:
- `yolo12n.pt` (nano, уже обучена)  
- `yolo12s.pt` (small)

Используйте те же гиперпараметры. Заполните таблицу:

| Модель | Параметры | mAP@0.5 | Время/img (ms) |
|:------:|:---------:|:-------:|:--------------:|
| YOLO12n | ? | ? | ? |
| YOLO12s | ? | ? | ? |

**Вопрос:** На сколько % улучшается mAP при увеличении модели?


In [ ]:
# TODO: обучите yolo12s и сравните с yolo12n

# model_yolo12s = YOLO("yolo12s.pt")
# results_yolo12s = model_yolo12s.train(
#     data    = str(data_yaml),
#     imgsz   = IMG_SIZE,
#     epochs  = EPOCHS,
#     batch   = BATCH,
#     lr0     = 0.001,
#     amp     = False,
#     workers = 0,
# )
# save_best_weights(results_yolo12s, 'yolo12s_best.pt')
# print(f"YOLO12s mAP@0.5: {results_yolo12s.box.map50*100:.2f}%")


### Задание 2 — YOLO-World: другой датасет

Попробуйте YOLO-World **без дообучения** на другом изображении.  
Найдите изображение с несколькими объектами и детектируйте их текстовыми запросами.

Примеры запросов:
- `["car", "truck", "bus", "person"]`
- `["dog", "cat", "bird"]`
- `["bottle", "cup", "phone"]`

Сравните точность детекции до и после fine-tune.


In [ ]:
# TODO: YOLO-World на произвольном изображении

# ваш URL или путь к изображению
# img_url = "https://..."

# model_world_test = YOLOWorld("yolov8s-world.pt")
# model_world_test.set_classes(["...", "...", "..."])  # ваши классы

# results_world_test = model_world_test.predict(img_url, conf=0.2)
# results_world_test[0].save("yolo_world_custom.jpg")

# plt.figure(figsize=(8,6))
# plt.imshow(cv2.imread("yolo_world_custom.jpg")[:,:,::-1])
# plt.axis('off'); plt.show()


### Задание 3 — RT-DETR vs YOLO12: анализ ошибок

Для каждой модели найдите по 3 примера, где:
1. Обе модели **корректно** детектируют объекты
2. Только **YOLO12 прав**, а RT-DETR ошибается
3. Только **RT-DETR прав**, а YOLO12 ошибается

Визуализируйте найденные случаи и прокомментируйте,  
в каких ситуациях каждая архитектура работает лучше.


In [ ]:
# TODO: анализ ошибок

# def find_disagreements(model1, model2, img_paths, conf=0.3, iou_thresh=0.5):
#     """Находит изображения, где модели расходятся."""
#     # Подсказка: сравните количество боксов и их координаты
#     disagreements = []
#     for p in img_paths:
#         res1 = model1.predict(str(p), conf=conf, verbose=False)[0]
#         res2 = model2.predict(str(p), conf=conf, verbose=False)[0]
#         n1, n2 = len(res1.boxes), len(res2.boxes)
#         if abs(n1 - n2) > 0:  # разное количество объектов
#             disagreements.append((p, n1, n2))
#     return disagreements

# if 'yolo12n_best.pt' in [str(p) for p in Path('.').glob('*.pt')] and \
#    'rtdetr_best.pt'  in [str(p) for p in Path('.').glob('*.pt')]:
#     m_yolo  = YOLO('yolo12n_best.pt')
#     m_rtdet = RTDETR('rtdetr_best.pt')
#     cases = find_disagreements(m_yolo, m_rtdet, test_imgs_list[:50])
#     print(f"Расхождений: {len(cases)}")


---
## 📋 Итоги воркшопа

| Модель | Тип | Плюсы | Минусы |
|:------:|:---:|:-----:|:------:|
| **YOLO12** | CNN | Очень быстрый, хорошая точность | Требует NMS |
| **RT-DETR** | Transformer | Без NMS, точнее на сложных сценах | Медленнее, больше параметров |
| **YOLO-World** | Open-vocab | Zero-shot, не нужен датасет | Ниже точность без fine-tune |

### Когда что использовать:
- **YOLO12n/s** → встроенные системы, real-time на CPU/GPU
- **RT-DETR** → когда важна максимальная точность и есть GPU
- **YOLO-World** → быстрый прототип, новые классы без разметки

### Полезные ссылки:
- [Ultralytics Docs](https://docs.ultralytics.com/)
- [RT-DETR Paper](https://arxiv.org/abs/2304.08069)
- [YOLO-World Paper](https://arxiv.org/abs/2401.17270)
- [Roboflow Datasets](https://universe.roboflow.com/)
